# Text to Image Diffusion Model

### Author : Kanika .

- First Prompt: "Generate one image per label (drink, food, inside, outside) using captions from a text-to-image diffusion model like GLIDE or Stable Diffusion."

- Last Prompt: "How to save and zip the model and generated images?”

### Install and Load Stable Diffusion Model

In [17]:
#pip install torch torchvision diffusers transformers scipy scikit-learn tqdm

In [18]:
#pip install diffusers transformers accelerate

In [3]:
# Image generation (GLIDE / Stable Diffusion)
from diffusers import StableDiffusionPipeline
from transformers import logging
import torch

# Evaluation: IS and FID
from torchvision.models import inception_v3
from torchvision import transforms
from torch.nn import functional as F
from PIL import Image
import numpy as np
from scipy.linalg import sqrtm
from sklearn.metrics import accuracy_score  # optional

# File handling and data ops
import os
import pandas as pd
from tqdm import tqdm

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import zipfile
zip_path = "/content/drive/MyDrive/Colab Notebooks/Ass-3/final_processed_data.zip"
extract_path = "/content/final_processed_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [7]:
# Load metadata
df = pd.read_csv("/content/final_processed_data/final_processed_data/metadata/train_metadata.csv", encoding='utf-8')

print(df.columns)

Index(['photo_id', 'path', 'absolute_path', 'label', 'business_id', 'caption',
       'split', 'file_size', 'modified_time'],
      dtype='object')


### Pick One Caption Per Class for Image Generation

In [9]:
# Select one caption per class
captions_per_class = df.groupby("label").apply(lambda x: x.sample(1)).reset_index(drop=True)

/tmp/ipython-input-9-1741812262.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  captions_per_class = df.groupby("label").apply(lambda x: x.sample(1)).reset_index(drop=True)


In [10]:
# Print captions you'll use
for idx, row in captions_per_class.iterrows():
    print(f"Class: {row['label']}\nCaption: {row['caption']}\n")

Class: drink
Caption: Everywhere

Class: food
Caption: The Works

Class: inside
Caption: nan

Class: outside
Caption: Courtyard at Bistro di Marino



In [11]:
df = df[df['caption'].notna()]  # Drop NaNs from caption before sampling

captions_per_class = df.groupby("label").apply(lambda x: x.sample(1)).reset_index(drop=True)

for idx, row in captions_per_class.iterrows():
    print(f"Class: {row['label']}\nCaption: {row['caption']}\n")

Class: drink
Caption: Iced Shaken Espresso

Class: food
Caption: It was as good as it looks.

Class: inside
Caption: The rush hour

Class: outside
Caption: Great new place



/tmp/ipython-input-11-3143374632.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  captions_per_class = df.groupby("label").apply(lambda x: x.sample(1)).reset_index(drop=True)


### Generate Image from Each Caption

In [12]:
# Directory to save your generated images
output_dir = "/content/generated_images"
os.makedirs(output_dir, exist_ok=True)

In [13]:
# Generate and save one image per label
for idx, row in captions_per_class.iterrows():
    label = row['label']
    caption = row['caption']

    print(f"🧠 Prompt: '{caption}' for class: {label}")
    image = pipe(caption).images[0]

    image_path = os.path.join(output_dir, f"{label}_gen.png")
    image.save(image_path)
    print(f"✅ Saved: {image_path}")

🧠 Prompt: 'Iced Shaken Espresso' for class: drink


  0%|          | 0/50 [00:00<?, ?it/s]

✅ Saved: /content/generated_images/drink_gen.png
🧠 Prompt: 'It was as good as it looks.' for class: food


  0%|          | 0/50 [00:00<?, ?it/s]

✅ Saved: /content/generated_images/food_gen.png
🧠 Prompt: 'The rush hour' for class: inside


  0%|          | 0/50 [00:00<?, ?it/s]

✅ Saved: /content/generated_images/inside_gen.png
🧠 Prompt: 'Great new place' for class: outside


  0%|          | 0/50 [00:00<?, ?it/s]

✅ Saved: /content/generated_images/outside_gen.png


In [15]:
import shutil

shutil.make_archive("/content/generated_images", 'zip', "/content/generated_images")
print("✅ Zipped: /content/generated_images.zip")

✅ Zipped: /content/generated_images.zip


In [14]:
model_save_path = "/content/saved_model"

pipe.save_pretrained(model_save_path)
print(f"✅ Model saved at: {model_save_path}")

✅ Model saved at: /content/saved_model


In [16]:
shutil.make_archive("/content/i2t_diffusion_model", 'zip', model_save_path)
print("✅ Zipped: /content/saved_model.zip")

✅ Zipped: /content/saved_model.zip


### Text-to-Image Diffusion Summary

This section implements a text-to-image generation pipeline using the Stable Diffusion model (v1.4) from HuggingFace.

- Captions were sampled from the training metadata (1 per label)
- Images generated for 4 labels: `drink`, `food`, `inside`, `outside`
- Model and outputs saved for reproducibility and evaluation

### Artifacts:
- `generated_images.zip`: 4 generated images
- `t2t_diffusion_model.zip`: Saved Stable Diffusion pipeline

Evaluation using FID and Inception Score is planned for synthetic vs real image comparison.